本 Notebook 使用离线构造数据。固定申报事实与图表属于历史教学材料；新计算不复现市场策略。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch10_markets_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第10章：金融市场工具与现金流

本 Notebook 全部使用构造输入，不含真实行情或投资建议。金额的币种、报价方向、合约乘数与事件时间按单元格注明。所有计算函数与脚本共用 `code/ch10_markets.py`。请从课程根目录启动并从头依次运行。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "code" / "ch10_markets.py").exists():
    raise FileNotFoundError("请从 quant_course 项目根目录启动 Notebook")
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from quant_course.market_instruments import (
    bond_coupon_cashflow,
    covered_fx_forward_quote,
    equity_total_return,
    fx_convert_base_to_quote,
    futures_variation_margin_usd,
    interest_rate_swap_net_cashflow,
    listed_option_pnl,
    repo_cash_advanced,
)
from ch10_markets import build_cashflow_timeline

## 股票与债券：权利形成和现金到账分开

股票构造：100 股，买入 USD 50.00/share，期末 USD 50.20/share，持有期间股息 USD 0.50/share。函数返回持有期简单总收益；它不模拟除息日、税或股息支付日。债券票息函数返回每次付息金额，面值和利率按合同单位输入。

In [ ]:
equity_return = equity_total_return(100, 50.00, 50.20, 0.50)
semiannual_coupon = bond_coupon_cashflow(1_000.0, 0.06, 2)
swap_cashflow = interest_rate_swap_net_cashflow(1_000_000.0, 0.04, 0.036, 0.25)
assert abs(equity_return - 0.014) < 1e-12
assert abs(semiannual_coupon - 30.0) < 1e-12
assert abs(swap_cashflow - (-1_000.0)) < 1e-12
print(f"股票简单总收益：{equity_return:.2%}")
print(f"半年票息：USD {semiannual_coupon:.2f}")
print(f"互换首期净现金流：USD {swap_cashflow:.2f}")

## FX：先读报价方向

`USD/EUR = 1.10` 表示每 1 EUR 报价 USD 1.10。远期函数只实现同到期日、简单利率、无摩擦条件下的参考关系。

In [ ]:
spot_usd_per_eur = 1.10
converted_usd = fx_convert_base_to_quote(10_000.0, spot_usd_per_eur)
forward_usd_per_eur = covered_fx_forward_quote(spot_usd_per_eur, 0.05, 0.03, 0.5)
assert converted_usd == 11_000.0
assert abs(forward_usd_per_eur - (1.10 * 1.025 / 1.015)) < 1e-12
print(f"EUR 10,000 -> USD {converted_usd:,.2f} at the stated spot")
print(f"simple parity reference: USD per EUR {forward_usd_per_eur:.6f}")

## MES、期权与 repo：现金流按合约单位换算

期货输入是连续结算间指数点数变化，乘数为 USD/index point。期权损益使用相同 USD/share 报价、显式乘数，并在到期 payoff 中扣开仓期权费。Repo 函数只演示 haircut 对融资额的算术关系。

In [ ]:
mes_vm = futures_variation_margin_usd(1, 5.0, [20.0, -40.0, 30.0])
call_pnl = listed_option_pnl(110.0, 100.0, 4.0, 1, 100, "call")
repo_advance = repo_cash_advanced(1_000_000.0, 0.05)
assert mes_vm == (100.0, -200.0, 150.0)
assert call_pnl == 600.0
assert repo_advance == 950_000.0
print(f"MES variation cash flows: {mes_vm}; net USD {sum(mes_vm):.2f}")
print(f"long call expiry P&L: USD {call_pnl:.2f}")
print(f"repo cash advanced: USD {repo_advance:,.2f}")

## 事件图

图的横向位置表示事件顺序，不是按日历比例缩放；纵向泳道表示工具分类而非金额大小。金额标签均为构造 USD。蓝色菱形是股息权利形成标记，不是当日收到现金。

In [ ]:
figure_path = build_cashflow_timeline(RUN_DIR / "figures/a3/ch10_cashflow_timeline.png")
print(f"wrote {figure_path.relative_to(PROJECT_ROOT)}")
from ch10_markets import build_cashflow_timeline